# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder, Seq2Seq(Encoder+Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while문)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### 1. 데이터 로드 및 정제

In [4]:
# df = pd.read_csv('/content/emotional_conversation_corpus.csv', encoding='cp949')
df = pd.read_csv('./data/emotional_conversation_corpus.csv', encoding='cp949')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51630 entries, 0 to 51629
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  51630 non-null  int64
 1   연령          51630 non-null  str  
 2   성별          51630 non-null  str  
 3   상황키워드       51630 non-null  str  
 4   신체질환        51630 non-null  str  
 5   감정_대분류      51630 non-null  str  
 6   감정_소분류      51630 non-null  str  
 7   사람문장1       51630 non-null  str  
 8   시스템문장1      51630 non-null  str  
 9   사람문장2       51630 non-null  str  
 10  시스템문장2      51630 non-null  str  
 11  사람문장3       42695 non-null  str  
 12  시스템문장3      42695 non-null  str  
dtypes: int64(1), str(12)
memory usage: 30.2 MB


In [5]:
df = df[df['연령'] == '청년']
df.info()

<class 'pandas.DataFrame'>
Index: 14832 entries, 0 to 51617
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  14832 non-null  int64
 1   연령          14832 non-null  str  
 2   성별          14832 non-null  str  
 3   상황키워드       14832 non-null  str  
 4   신체질환        14832 non-null  str  
 5   감정_대분류      14832 non-null  str  
 6   감정_소분류      14832 non-null  str  
 7   사람문장1       14832 non-null  str  
 8   시스템문장1      14832 non-null  str  
 9   사람문장2       14832 non-null  str  
 10  시스템문장2      14832 non-null  str  
 11  사람문장3       11734 non-null  str  
 12  시스템문장3      11734 non-null  str  
dtypes: int64(1), str(12)
memory usage: 8.7 MB


In [6]:
# 컬럼명 변경
df_qa1 = df[['사람문장1', '시스템문장1']]
df_qa1.columns = ['question', 'answer']
df_qa1.isna().sum()

question    0
answer      0
dtype: int64

In [7]:
df_qa2 = df[['사람문장2', '시스템문장2']]
df_qa2.columns = ['question', 'answer']
df_qa2.isna().sum()

question    0
answer      0
dtype: int64

In [8]:
df_qa3 = df[['사람문장3', '시스템문장3']]
df_qa3.columns = ['question', 'answer']
df_qa3.isna().sum()

question    3098
answer      3098
dtype: int64

In [9]:
# 결측치 제거
df_qa3 = df_qa3.dropna(how='any')
df_qa3.isna().sum()

question    0
answer      0
dtype: int64

In [10]:
# 데이터 병합
df_qa1 = pd.concat([df_qa1, df_qa2], axis=0)
df_qa1.info()

<class 'pandas.DataFrame'>
Index: 29664 entries, 0 to 51617
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   question  29664 non-null  str  
 1   answer    29664 non-null  str  
dtypes: str(2)
memory usage: 5.3 MB


In [11]:
df_qa1 = pd.concat([df_qa1, df_qa3], axis=0)
df_qa1.info()

<class 'pandas.DataFrame'>
Index: 41398 entries, 0 to 51617
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   question  41398 non-null  str  
 1   answer    41398 non-null  str  
dtypes: str(2)
memory usage: 7.2 MB


In [12]:
# 인덱스 초기화
df_qa1 = df_qa1.reset_index(drop=True)
df_qa1.info()

<class 'pandas.DataFrame'>
RangeIndex: 41398 entries, 0 to 41397
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   question  41398 non-null  str  
 1   answer    41398 non-null  str  
dtypes: str(2)
memory usage: 6.9 MB


### 2. 학습 데이터 준비

In [13]:
ques_inputs = []
ans_inputs = []
ans_targets = []

for i in range(41398):
    ques_inputs.append(df_qa1['question'][i])
    ans_inputs.append('<sos> ' + df_qa1['answer'][i])
    ans_targets.append(df_qa1['answer'][i] + ' <eos>')

In [14]:
print(len(ques_inputs), len(ans_inputs), len(ans_targets))
print(ques_inputs[7777])
print(ans_inputs[7777])

41398 41398 41398
드디어 내가 꿈에 그리던 회사에 합격했어. 기분이 너무 좋아.
<sos> 가고 싶던 곳에 갈 수 있게 되어 무척 행복하시겠어요.


##### (+) 전역변수 선언

In [15]:
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
LATENT_DIM = 512

### 3. 데이터 전처리

In [16]:
# 토큰화
ques_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
ques_tokenizer.fit_on_texts(ques_inputs)
ques_seqs = ques_tokenizer.texts_to_sequences(ques_inputs)

ques_num_words = len(ques_tokenizer.word_index) + 1
ques_max_len = max(len(s) for s in ques_seqs)

print(f'{ques_num_words = }')
print(f'{ques_max_len = }')

ques_num_words = 47128
ques_max_len = 29


In [17]:
ans_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
ans_tokenizer.fit_on_texts(ans_inputs + ans_targets)

ans_input_seqs = ans_tokenizer.texts_to_sequences(ans_inputs)
ans_target_seqs = ans_tokenizer.texts_to_sequences(ans_targets)

ans_num_words = len(ans_tokenizer.word_index) + 1
ans_max_len = max(len(s) for s in ans_input_seqs)

print(f'{ans_num_words = }')
print(f'{ans_max_len = }')

ans_num_words = 34213
ans_max_len = 27


In [18]:
# 패딩 처리
encoder_inputs = pad_sequences(ques_seqs, maxlen=ques_max_len, padding='pre')
decoder_inputs = pad_sequences(ans_input_seqs, maxlen=ans_max_len, padding='post')
decoder_targets = pad_sequences(ans_target_seqs, maxlen=ans_max_len, padding='post')

print(encoder_inputs.shape)
print(decoder_inputs.shape)
print(decoder_targets.shape)

print(encoder_inputs[1000])
print([ques_tokenizer.index_word[s] for s in encoder_inputs[1000] if s != 0])
print(decoder_inputs[1000])
print([ans_tokenizer.index_word[s] for s in decoder_inputs[1000] if s != 0])
print(decoder_targets[1000])
print([ans_tokenizer.index_word[s] for s in decoder_targets[1000] if s != 0])

(41398, 29)
(41398, 27)
(41398, 27)
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0  67 444 168  69  63  31  10   1 126]
['회사', '동기가', '어제', '했어', '같이', '한', '게', '너무', '속상해']
[   1 1683 2965  994 1103   23  244   35   21 2560   60    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0]
['<sos>', '동기와', '약속한', '지', '얼마', '안', '돼서', '그런', '일이', '있었다니', '속상하시겠어요.']
[1683 2965  994 1103   23  244   35   21 2560   60    2    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0]
['동기와', '약속한', '지', '얼마', '안', '돼서', '그런', '일이', '있었다니', '속상하시겠어요.', '<eos>']


In [19]:
# 데이터 로더 설정
class ECCDataset(Dataset):
  def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
    super().__init__()
    self.encoder_inputs = encoder_inputs
    self.decoder_inputs = decoder_inputs
    self.decoder_targets = decoder_targets

  def __len__(self):
    return len(self.encoder_inputs)

  def __getitem__(self, index):
    return (
        torch.tensor(self.encoder_inputs[index], dtype = torch.long),
        torch.tensor(self.decoder_inputs[index], dtype = torch.long),
        torch.tensor(self.decoder_targets[index], dtype = torch.long)
    )

In [20]:
train_index, val_index = train_test_split(range(len(encoder_inputs)), random_state=0)
print(len(train_index), len(val_index))

train_dataset = ECCDataset(
    encoder_inputs[train_index],
    decoder_inputs[train_index],
    decoder_targets[train_index]
)
val_dataset = ECCDataset(
   encoder_inputs[val_index],
   decoder_inputs[val_index],
   decoder_targets[val_index]
)

31048 10350


In [21]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 4. 모델 준비

##### 인코더(Encoder)

In [22]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

##### 디코더(Decoder)

In [23]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s

##### Seq2Seq

In [24]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output

In [25]:
encoder = Encoder(ques_num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(ans_num_words, EMBEDDING_DIM, LATENT_DIM)

model = Seq2Seq(encoder, decoder)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(47128, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(34213, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=34213, bias=True)
  )
)

### 5. 모델 학습

In [ ]:
# 이미 학습하여 pth 파일로 모델을 저장하였으므로 '5. 모델 학습' 해당 셀은 실행시키지 않아도 된다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(ques_num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(ans_num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)
    output = output.view(-1, output.size(-1))    # output과 dec_targets의 크기 맞춰줌
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  #------------------------------------------------

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))    # output과 dec_targets의 크기 맞춰줌
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/100 TrainLoss=6.3395 TrainAcc=0.1875 ValLoss=5.7065 ValAcc=0.2249
Epoch 2/100 TrainLoss=5.3199 TrainAcc=0.2466 ValLoss=5.2693 ValAcc=0.2602
Epoch 3/100 TrainLoss=4.7595 TrainAcc=0.2773 ValLoss=5.0547 ValAcc=0.2772
Epoch 4/100 TrainLoss=4.2978 TrainAcc=0.3006 ValLoss=4.9701 ValAcc=0.2866
Epoch 5/100 TrainLoss=3.8935 TrainAcc=0.3247 ValLoss=4.9403 ValAcc=0.2919
Epoch 6/100 TrainLoss=3.5176 TrainAcc=0.3588 ValLoss=4.9641 ValAcc=0.2953
Epoch 7/100 TrainLoss=3.1697 TrainAcc=0.4027 ValLoss=5.0046 ValAcc=0.2967
Epoch 8/100 TrainLoss=2.8515 TrainAcc=0.4500 ValLoss=5.0774 ValAcc=0.2950
Epoch 9/100 TrainLoss=2.5598 TrainAcc=0.4989 ValLoss=5.1497 ValAcc=0.2962
Epoch 10/100 TrainLoss=2.2850 TrainAcc=0.5472 ValLoss=5.2343 ValAcc=0.2946
Epoch 11/100 TrainLoss=2.0266 TrainAcc=0.5964 ValLoss=5.3431 ValAcc=0.2908
Epoch 12/100 TrainLoss=1.7843 TrainAcc=0.6455 ValLoss=5.4488 ValAcc=0.2897
Epoch 13/100 TrainLoss=1.5571 TrainAcc=0.6926 ValLoss=5.5840 ValAcc=0.2883
Epoch 14/100 TrainLoss=1.3450 Trai

In [63]:
# 모델 저장
torch.save(model, 'seq2seq_teacher_forcing_ECC.pth')

In [27]:
# 모델 불러오기
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = torch.load('./data/seq2seq_teacher_forcing_ECC.pth', 
                   map_location=device,  # 현재 환경(CPU/GPU)에 맞춰 자동으로 매핑
                   weights_only=False)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(47128, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(34213, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=34213, bias=True)
  )
)

### 6. 모델 추론

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def answer(input_seq, model, ans_tokenizer, max_len=ans_max_len, device=device):
  model = model.to(device)
  model.eval()
  encoder = model.encoder
  decoder = model.decoder

  input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

  # Encoder 처리
  with torch.no_grad():
    hidden, cell = encoder(input_seq)

  # Decoder 출력(Auto Regressive)
  sos_index = ans_tokenizer.word_index['<sos>']
  eos_index = ans_tokenizer.word_index['<eos>']

  output_sentences = []

  target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

  with torch.no_grad():
    for _ in range(max_len):
      output, hidden, cell = decoder(target_seq, hidden, cell)  # (batch_size, seq_len, vocab_size)
      proba = output.squeeze(1).softmax(dim=-1)                 # (batch_size, vocab_size)
      pred = proba.argmax(dim=-1).detach().cpu().item()

      if pred == eos_index:
        break

      if pred > 0:
        word = ans_tokenizer.index_word[pred]
        output_sentences.append(word)

      target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

  return ' '.join(output_sentences)

In [29]:
for _ in range(5):
  index = np.random.choice(len(ques_inputs))
  input_seq = encoder_inputs[index:index+1]
  output = answer(input_seq, model, ans_tokenizer)

  print(f'질문: {ques_inputs[index]}')
  print(f'답변: {ans_inputs[index]}')
  print(f'모델 추론 답변: {output}')
  print()

질문: 취직을 하고 싶어도 취직도 안 돼.
답변: <sos> 취직이 안되서 괴로우시군요.
모델 추론 답변: 취직이 안되서 괴로우시군요.

질문: 호의를 이젠 권리로 알고 나를 막 대하고 있어.
답변: <sos> 앞으로 어떻게 하실 계획인가요?
모델 추론 답변: 좋은 친구들을 만나면 곁에 계신 걸 정말 다행이네요.

질문: 남자친구를 밖에서?한 시간을?기다렸어.
답변: <sos> 요새처럼 더운 날씨에 정말 고생하셨겠어요.
모델 추론 답변: 어떻게 하면 팀장님의 업무를 있을까요?

질문: 현직자를 만나보면서 진로 결정을 빨리해야 할 것 같아.
답변: <sos> 현직자를 만나보면서 빠르게 진로를 결정하려고 하시는군요.
모델 추론 답변: 어떻게 하면 다시 일할 수 있을까요?

질문: 그동안 못 했던 게임도 하고 친구들도 만나야지.
답변: <sos> 그동안 못 했던 것들을 하면서 즐거운 시간 되셨으면 좋겠어요.
모델 추론 답변: 그동안 못 했던 것들을 하면서 즐거운 시간 되셨으면 좋겠어요.



In [36]:
def answer_to_user(ques, ques_tokenizer=ques_tokenizer):
  input_seq = ques_tokenizer.texts_to_sequences([ques])
  input_seq = pad_sequences(input_seq, maxlen=ques_max_len, padding='pre')
  return answer(input_seq, model, ans_tokenizer)

ques_texts = [
    '나 정말 열심히 면접 준비해서 붙을 줄 알았는데 탈락해서 너무 당황스러워.',
    '자랑만 하는 친구에게 듣기 불편하다고 말할까?',
    '서울로 이사를 왔더니 아는 사람이 없어서 외로워.',
    '나는 남자친구와 함께 있으면 세상에서 가장 행복한 사람이 되는 것 같아.'
]

for ques in ques_texts:
  ans = answer_to_user(ques)
  print(f'{ques} => {ans}')

나 정말 열심히 면접 준비해서 붙을 줄 알았는데 탈락해서 너무 당황스러워. => 열심히 준비한 프로젝트가 실패해 불안하시군요.
자랑만 하는 친구에게 듣기 불편하다고 말할까? => 친구에게 직접 생각을 해결해 보려 하시는군요.
서울로 이사를 왔더니 아는 사람이 없어서 외로워. => 함께 있어 줄 사람이 곁에 있었으면 좋겠네요.
나는 남자친구와 함께 있으면 세상에서 가장 행복한 사람이 되는 것 같아. => 남자친구와 함께 있으면 행복한 결혼생활을 유지할 수 있다고 생각하시는군요.


### 7. 챗봇 활용

In [34]:
while True:
    user_input = input("하고 싶은 말씀이 있으신가요? 없으시면 '종료'를 입력해 주세요.")
    if user_input == '종료':
        break

    chatbot_answer = answer_to_user(user_input)

    print('사용자 입력 내용 :', user_input)
    print('ECC 챗봇의 답변 :', chatbot_answer)

print('ECC 챗봇을 종료합니다.')

사용자 입력 내용 : 이따 저녁 뭐 먹지?
ECC 챗봇의 답변 : 여자 친구를 정말 기분 좋으시겠어요.
사용자 입력 내용 : 이따 발표 있는데 긴장 돼
ECC 챗봇의 답변 : 이런 일이 있을 때 어떻게 대처하셨어요?
사용자 입력 내용 : 연습 많이 해야겠지 뭐
ECC 챗봇의 답변 : 저도 같은 마음 충분히 자꾸 결혼할 수 없을 것 같아요.
ECC 챗봇을 종료합니다.
